# DeBERTa-v3-Large Crypto Classifier (Colab)

Steps:
1. Runtime → Change runtime type → GPU.
2. Upload datasets to `/content/`:
   - `crypto_twitter_dataset2.csv` (positive, preferred)
   - `non_crypto_tweets.csv` (negative)
   - optional fallback: `trashed_crypto_dataset.csv`
3. Run cells top to bottom.

Data format:
- Positive CSV can be `tweet_text` or `text`.
- Negative CSV uses `full_text` (semicolon-separated).

Notes:
- Main model is fixed to `microsoft/deberta-v3-large`.
- Pipeline removes contradictory cross-label duplicates.
- If training is too slow, reduce `MAX_SAMPLES_PER_CLASS` and/or `EPOCHS`.

In [ ]:
# Install dependencies (Colab)
!pip -q install -U transformers datasets accelerate scikit-learn matplotlib

In [ ]:
# Imports
import os
import random
import re
import html
import csv
import math
import warnings
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

warnings.filterwarnings("ignore", category=FutureWarning)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# Config (stability-first, with preferred positive dataset v2)
POS_PATH = "/content/crypto_twitter_dataset2.csv"
NEG_PATH = "/content/non_crypto_tweets.csv"
OUT_ROOT = "/content/models/crypto-detector-deberta-v3-large"

MODEL_NAME = "microsoft/deberta-v3-large"

SEED = 42
MAX_LEN = 128
EPOCHS = 2
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 8e-6
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 1
LABEL_SMOOTHING = 0.0
SCHEDULER_TYPE = "cosine"
GRADIENT_CHECKPOINTING = False

# Keep full precision first to avoid silent NaN on some T4 runs.
# Options: "no" | "fp16" | "bf16"
USE_MIXED_PRECISION = "no"

# Use all available balanced data (minority class cap)
MAX_SAMPLES_PER_CLASS = None

if USE_MIXED_PRECISION == "bf16" and torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    FP16, BF16 = False, True
elif USE_MIXED_PRECISION == "fp16" and torch.cuda.is_available():
    FP16, BF16 = True, False
else:
    FP16, BF16 = False, False

os.makedirs(OUT_ROOT, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True

print(f"Model={MODEL_NAME}")
print(f"POS_PATH={POS_PATH}")
print(f"NEG_PATH={NEG_PATH}")
print(f"Precision -> fp16={FP16}, bf16={BF16}")
print(f"MAX_SAMPLES_PER_CLASS={MAX_SAMPLES_PER_CLASS}")

In [ ]:
# Helper functions (quality-controlled)

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _light_normalize(s: str) -> str:
    s = s.lower()
    s = re.sub(r"https?://\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    s = re.sub(r"#\w+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _is_low_info_text(s: str) -> bool:
    if not s:
        return True
    t = _light_normalize(s)
    t2 = re.sub(r"[^A-Za-zА-Яа-я0-9 ]", " ", t)
    t2 = re.sub(r"\s+", " ", t2).strip()
    # Reject ultra-short, URL-only, or mostly-symbolic content
    if len(t2) < 12:
        return True
    if sum(ch.isalnum() for ch in t2) < 8:
        return True
    return False


def robust_read_pos(path: str) -> pd.DataFrame:
    attempts = [
        {"sep": ",", "engine": "python", "on_bad_lines": "skip", "encoding_errors": "replace"},
        {"sep": None, "engine": "python", "on_bad_lines": "skip", "encoding_errors": "replace"},
    ]
    last_err = None
    for kw in attempts:
        try:
            df = pd.read_csv(path, **kw)
            if len(df) > 0 and len(df.columns) > 0:
                return df
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Failed to read POS csv: {last_err}")


def robust_read_neg(path: str) -> pd.DataFrame:
    attempts = [
        {"sep": ";", "engine": "python", "on_bad_lines": "skip", "encoding_errors": "replace"},
        {"sep": None, "engine": "python", "on_bad_lines": "skip", "encoding_errors": "replace"},
    ]
    last_err = None
    for kw in attempts:
        try:
            df = pd.read_csv(path, **kw)
            if len(df) > 0 and len(df.columns) > 0:
                return df
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Failed to read NEG csv: {last_err}")


def _normalize_text_df(df: pd.DataFrame, text_col: str, label: int) -> pd.DataFrame:
    out = df[[text_col]].rename(columns={text_col: "text"}).copy()
    out = out.dropna(subset=["text"]).copy()
    out["text"] = out["text"].astype(str).apply(clean_text)
    out = out[out["text"].ne("")]
    out = out[~out["text"].apply(_is_low_info_text)]
    out["text_norm"] = out["text"].apply(_light_normalize)
    out = out.drop_duplicates(subset=["text_norm"]).reset_index(drop=True)
    out["label"] = int(label)
    return out[["text", "text_norm", "label"]]


def load_positive(path: str) -> pd.DataFrame:
    df = robust_read_pos(path)
    col = next((c for c in ["tweet_text", "text", "full_text", "clean_text"] if c in df.columns), None)
    if col is None:
        raise ValueError(f"Positive text column not found. Columns: {list(df.columns)}")
    return _normalize_text_df(df, col, 1)


def load_negative(path: str) -> pd.DataFrame:
    df = robust_read_neg(path)
    col = next((c for c in ["full_text", "text", "tweet_text", "clean_text"] if c in df.columns), None)
    if col is None:
        raise ValueError(f"Negative text column not found. Columns: {list(df.columns)}")
    return _normalize_text_df(df, col, 0)


def drop_cross_label_collisions(pos: pd.DataFrame, neg: pd.DataFrame):
    # Remove contradictory duplicates by normalized text.
    overlap = set(pos["text_norm"]).intersection(set(neg["text_norm"]))
    if not overlap:
        return pos, neg, 0

    pos2 = pos[~pos["text_norm"].isin(overlap)].reset_index(drop=True)
    neg2 = neg[~neg["text_norm"].isin(overlap)].reset_index(drop=True)
    return pos2, neg2, len(overlap)


def make_splits(pos: pd.DataFrame, neg: pd.DataFrame) -> DatasetDict:
    pos, neg, overlap_n = drop_cross_label_collisions(pos, neg)
    print(f"Cross-label overlaps removed: {overlap_n}")

    n = min(len(pos), len(neg))
    if MAX_SAMPLES_PER_CLASS is not None:
        n = min(n, int(MAX_SAMPLES_PER_CLASS))
    if n < 200:
        raise ValueError(f"Too few samples per class after cleaning: {n}")

    pos_s = pos.sample(n, random_state=SEED).reset_index(drop=True)
    neg_s = neg.sample(n, random_state=SEED).reset_index(drop=True)
    all_df = pd.concat([pos_s, neg_s], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

    train_df, temp_df = train_test_split(
        all_df, test_size=0.2, random_state=SEED, stratify=all_df["label"]
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["label"]
    )

    for d in (train_df, val_df, test_df):
        d["label"] = d["label"].astype(np.int64)

    print(f"Per-class={n}; train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
    print("Train labels:", train_df["label"].value_counts().to_dict())
    print("Val labels:", val_df["label"].value_counts().to_dict())
    print("Test labels:", test_df["label"].value_counts().to_dict())

    return DatasetDict({
        "train": Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True)),
        "val": Dataset.from_pandas(val_df[["text", "label"]].reset_index(drop=True)),
        "test": Dataset.from_pandas(test_df[["text", "label"]].reset_index(drop=True)),
    })


def tokenize_dataset(ds: DatasetDict, tokenizer: AutoTokenizer) -> DatasetDict:
    def tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN, padding=False)

    mapped = ds.map(tok, batched=True)
    keep = {"input_ids", "attention_mask", "token_type_ids", "label"}
    for split in ["train", "val", "test"]:
        drop_cols = [c for c in mapped[split].column_names if c not in keep]
        if drop_cols:
            mapped[split] = mapped[split].remove_columns(drop_cols)
    return mapped


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = (preds == labels).mean()
    return {"accuracy": float(acc), "precision": float(precision), "recall": float(recall), "f1": float(f1)}

In [ ]:
# Inspect and normalize datasets
pos_raw = robust_read_pos(POS_PATH)
neg_raw = robust_read_neg(NEG_PATH)

print("POS columns:", list(pos_raw.columns))
print("NEG columns:", list(neg_raw.columns))

pos_text_col = next((c for c in ["tweet_text", "text", "full_text", "clean_text"] if c in pos_raw.columns), None)
neg_text_col = next((c for c in ["full_text", "text", "tweet_text", "clean_text"] if c in neg_raw.columns), None)
if pos_text_col is None or neg_text_col is None:
    raise ValueError("Required text columns not found")

pos_clean = _normalize_text_df(pos_raw, pos_text_col, label=1)
neg_clean = _normalize_text_df(neg_raw, neg_text_col, label=0)

overlap_count = len(set(pos_clean["text_norm"]).intersection(set(neg_clean["text_norm"])))
print("Cleaned sizes -> pos:", len(pos_clean), "neg:", len(neg_clean))
print("Cross-label overlap BEFORE removal:", overlap_count)
print("POS sample:", pos_clean[["text","label"]].head(2).to_dict())
print("NEG sample:", neg_clean[["text","label"]].head(2).to_dict())

pos_out = "/content/crypto_pos_normalized.csv"
neg_out = "/content/crypto_neg_normalized.csv"
pos_clean[["text","label"]].to_csv(pos_out, index=False)
neg_clean[["text","label"]].to_csv(neg_out, index=False)
print(f"Saved normalized files: {pos_out}, {neg_out}")

In [ ]:
# Train DeBERTa-v3-large (debug-safe pipeline)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

pos = load_positive(POS_PATH)
neg = load_negative(NEG_PATH)
print(f"Pos: {len(pos)}, Neg: {len(neg)}")

ds = make_splits(pos, neg)
ds_tok = tokenize_dataset(ds, tokenizer)
collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

# Dataset integrity checks (critical)
for split in ["train", "val", "test"]:
    labels_arr = np.array(ds_tok[split]["label"])
    uniq = sorted(np.unique(labels_arr).tolist())
    print(f"{split} unique labels:", uniq, "size:", len(labels_arr))
    if uniq != [0, 1]:
        raise ValueError(f"{split} labels are not binary [0,1]: {uniq}")

# Batch sanity check
batch = collator([ds_tok["train"][i] for i in range(min(8, len(ds_tok["train"])) )])
print("Batch keys:", list(batch.keys()))
print("Batch labels sample:", batch["labels"][:8].tolist())

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
    problem_type="single_label_classification",
)
model.config.use_cache = False
model.config.id2label = {0: "non_crypto", 1: "crypto"}
model.config.label2id = {"non_crypto": 0, "crypto": 1}

# Forward sanity check (train + val)
for split in ["train", "val"]:
    b = collator([ds_tok[split][i] for i in range(min(8, len(ds_tok[split])))])
    b = {k: v.to(model.device) for k, v in b.items()}
    with torch.no_grad():
        o = model(**b)
    print(f"{split} sanity loss:", float(o.loss), "| finite logits:", bool(torch.isfinite(o.logits).all().item()))

# Mini-train probe to catch NaN before full trainer
model.train()
probe_losses = []
for i in range(min(20, len(ds_tok["train"]))):
    b = collator([ds_tok["train"][i]])
    b = {k: v.to(model.device) for k, v in b.items()}
    o = model(**b)
    loss_val = float(o.loss.detach().cpu())
    if not np.isfinite(loss_val):
        raise RuntimeError(f"Non-finite loss detected in probe at i={i}: {loss_val}")
    probe_losses.append(loss_val)
print("Probe loss mean:", float(np.mean(probe_losses)))

steps_per_epoch = math.ceil(len(ds_tok["train"]) / (BATCH_SIZE * GRAD_ACCUM_STEPS))
total_steps = max(steps_per_epoch * EPOCHS, 1)
warmup_steps = int(total_steps * WARMUP_RATIO)

args = TrainingArguments(
    output_dir=OUT_ROOT,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    fp16=FP16,
    bf16=BF16,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=100,
    logging_nan_inf_filter=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to=[],
    label_smoothing_factor=LABEL_SMOOTHING,
    lr_scheduler_type=SCHEDULER_TYPE,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    optim="adamw_torch",
    dataloader_num_workers=2,
    remove_unused_columns=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["val"],
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

train_result = trainer.train()
val_metrics = trainer.evaluate()
test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])

print("Validation metrics:", val_metrics)
print("Test metrics:", test_metrics)

best_model_dir = OUT_ROOT
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)
print("Saved model to:", best_model_dir)

In [ ]:
# Plot confusion matrix on test set
import matplotlib.pyplot as plt

pred = trainer.predict(ds_tok["test"])
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=-1)
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(4,4))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix (test)")
plt.colorbar()
plt.xticks([0,1], ["non-crypto", "crypto"])
plt.yticks([0,1], ["non-crypto", "crypto"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
# Basic tests
assert len(ds_tok["train"]) > 0 and len(ds_tok["test"]) > 0
assert "input_ids" in ds_tok["train"].column_names
assert "label" in ds_tok["train"].column_names
print("Dataset/tokenization tests passed.")

In [ ]:
# Inference helper
from transformers import TextClassificationPipeline

best_model = AutoModelForSequenceClassification.from_pretrained(OUT_ROOT)
best_tokenizer = AutoTokenizer.from_pretrained(OUT_ROOT)

pipe = TextClassificationPipeline(
    model=best_model,
    tokenizer=best_tokenizer,
    return_all_scores=True,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
    max_length=MAX_LEN,
 )

examples = [
    "Bitcoin rally is breaking all records",
    "Someone tell me we will see this version again",
    "Massive airdrop for DeFi users ends tomorrow",
]

for ex in examples:
    out = pipe(ex)
    print("TEXT:", ex)
    print("PRED:", out)
    print("-"*60)